In [ ]:
import os

# Fix wandb — tránh bị hỏi interactive login
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_START_METHOD"] = "thread"

# Clone repo và cài đặt
!git clone https://github.com/minhvuongle2004/lung-diagnosis.git
%cd /kaggle/working/lung-diagnosis/ldct-benchmark
!pip install -e . -q

print("✅ Setup done!")

In [ ]:
import os

# Bước 1: Liệt kê toàn bộ datasets có trong /kaggle/input/
print("=== /kaggle/input/ contents ===")
for item in os.listdir("/kaggle/input"):
    print(f"  {item}/")

# Bước 2: Tự động tìm thư mục cha chứa LDCT-and-Projection-data
print("\n=== Tìm LDCT-and-Projection-data ===")
data_path = None

for dataset_slug in os.listdir("/kaggle/input"):
    base = f"/kaggle/input/{dataset_slug}"
    for root, dirs, files in os.walk(base):
        if "LDCT-and-Projection-data" in dirs:
            data_path = root
            sub = os.path.join(root, "LDCT-and-Projection-data")
            patients = sorted(os.listdir(sub))
            print(f"✅ Datafolder found: {data_path}")
            print(f"   Dataset slug: {dataset_slug}")
            print(f"   Số bệnh nhân: {len(patients)}")
            print(f"   5 đầu: {patients[:5]}")
            break
    if data_path:
        break

if data_path is None:
    print("❌ Không tìm thấy LDCT-and-Projection-data!")
    print("In cấu trúc chi tiết hơn:")
    for slug in os.listdir("/kaggle/input"):
        base = f"/kaggle/input/{slug}"
        for root, dirs, files in os.walk(base):
            depth = root.replace(base, "").count("/")
            if depth <= 2:
                print("  " * depth + f"{slug}/" + root.replace(base, ""))
            if depth >= 2:
                dirs.clear()

In [ ]:
import os
os.environ["WANDB_MODE"] = "offline"

# Kiểm tra data_path hợp lệ
assert data_path is not None, "❌ data_path=None! Chạy Cell 2 trước và đọc output."
ldct_dir = os.path.join(data_path, "LDCT-and-Projection-data")
assert os.path.exists(ldct_dir), f"❌ Không tồn tại: {ldct_dir}"
print(f"✅ Datafolder hợp lệ: {data_path}")

# Tạo config
config = f"""trainer: edrrednet
seed: 1339
datafolder: {data_path}
optimizer: adam
lr: 9.583e-05
adam_b1: 0.9
adam_b2: 0.999
loss_alpha: 0.1
loss_beta: 0.0
loss_gamma: 0.0
num_edge_blocks: 2
mbs: 16
max_iterations: 92994
data_subset: 1.0
patchsize: 128
iterations_before_val: 500
valsamples: 8
data_norm: meanstd
num_workers: 2
cuda: true
devices: 0
"""

with open("configs/edrrednet_kaggle.yaml", "w", encoding="utf-8") as f:
    f.write(config)

print(f"Config saved. Bắt đầu training seed 1339...")
!python -m ldctbench.scripts.train --config configs/edrrednet_kaggle.yaml

In [ ]:
import glob, shutil, os

output_dir = "/kaggle/working"
seed = 1339  # đổi thành 2024 hoặc 42 cho seed khác

# Tìm và copy checkpoint
checkpoints = glob.glob("wandb/latest-run/*.pt")
print(f"Checkpoints found: {checkpoints}")
for ckpt in checkpoints:
    dest = os.path.join(output_dir, f"seed{seed}_{os.path.basename(ckpt)}")
    shutil.copy(ckpt, dest)
    print(f"✅ Saved: {dest}")

# Copy train log
logs = glob.glob("wandb/latest-run/files/*.csv")
for log in logs:
    dest = os.path.join(output_dir, f"seed{seed}_{os.path.basename(log)}")
    shutil.copy(log, dest)
    print(f"✅ Log saved: {dest}")

print("\n✅ Done! Click 'Save Version' để lưu output.")